In [1]:
import json
import numpy as np

# Read the JSONL file
file_path = "/home/monoshi/CodeSemantic/CodeSemantic/dataset/input_output_dataset_python.jsonl"
code_lengths = []

with open(file_path, 'r') as file:
    for line in file:
        try:
            data = json.loads(line.strip())
            code_lengths.append(data["code_length"])
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Error parsing line: {e}")
            continue

# Calculate statistics
if code_lengths:
    code_lengths_sorted = sorted(code_lengths)
    total_samples = len(code_lengths)
    
    # Calculate percentile boundaries
    p33 = np.percentile(code_lengths, 33.33)
    p66 = np.percentile(code_lengths, 66.66)
    
    min_length = min(code_lengths)
    max_length = max(code_lengths)
    avg_length = np.mean(code_lengths)
    median_length = np.median(code_lengths)
    
    print(f"Dataset Statistics:")
    print(f"Total samples: {total_samples}")
    print(f"Min code length: {min_length}")
    print(f"Max code length: {max_length}")
    print(f"Average code length: {avg_length:.2f}")
    print(f"Median code length: {median_length}")
    print()
    print(f"Code Length Distribution (33.3% percentiles):")
    print(f"Lower 33%:  {min_length:.0f} - {p33:.0f} lines of code")
    print(f"Middle 33%: {p33:.0f} - {p66:.0f} lines of code")
    print(f"Upper 33%:  {p66:.0f} - {max_length:.0f} lines of code")
    
    # Count samples in each range
    lower_count = len([x for x in code_lengths if x <= p33])
    middle_count = len([x for x in code_lengths if p33 < x <= p66])
    upper_count = len([x for x in code_lengths if x > p66])
    
    print()
    print(f"Sample distribution:")
    print(f"Lower 33%:  {lower_count} samples ({lower_count/total_samples*100:.1f}%)")
    print(f"Middle 33%: {middle_count} samples ({middle_count/total_samples*100:.1f}%)")
    print(f"Upper 33%:  {upper_count} samples ({upper_count/total_samples*100:.1f}%)")
    
else:
    print("No valid data found in the file.")

Dataset Statistics:
Total samples: 308
Min code length: 2
Max code length: 82
Average code length: 12.90
Median code length: 9.0

Code Length Distribution (33.3% percentiles):
Lower 33%:  2 - 7 lines of code
Middle 33%: 7 - 14 lines of code
Upper 33%:  14 - 82 lines of code

Sample distribution:
Lower 33%:  119 samples (38.6%)
Middle 33%: 93 samples (30.2%)
Upper 33%:  96 samples (31.2%)


In [2]:
import json
import re

def count_time_module_api_calls(dataset_path):
    """
    Count how many samples with Statement Type "API" have time module functions in "selected statement"
    """
    time_module_functions = {
        'time', 'sleep', 'ctime', 'gmtime', 'localtime', 'mktime', 'strftime', 'strptime',
        'asctime', 'perf_counter', 'process_time', 'time_ns', 'monotonic', 'thread_time'
    }
    
    time_pattern = re.compile(r'\btime\.(\w+)\b')
    
    total_api_calls = 0
    time_module_api_calls = 0
    matching_samples = []
    
    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    
                    # Check if statement type is API
                    if data.get("Statement Type") == "API":
                        total_api_calls += 1
                        
                        selected_statement = data.get("Selected Statement", "")
                        
                        # Check for time module function calls
                        matches = time_pattern.findall(selected_statement)
                        if matches:
                            time_module_api_calls += 1
                            matching_samples.append({
                                'selected_statement': selected_statement,
                                'time_functions': matches,
                                'idx': data.get('idx')
                            })
                        
                        # Also check for direct function names without module prefix
                        for func in time_module_functions:
                            if re.search(r'\b' + re.escape(func) + r'\s*\(', selected_statement):
                                if not any(m['idx'] == data.get('idx') for m in matching_samples):
                                    time_module_api_calls += 1
                                    matching_samples.append({
                                        'selected_statement': selected_statement,
                                        'time_functions': [func],
                                        'idx': data.get('idx')
                                    })
                                break
                                
                except json.JSONDecodeError as e:
                    print(f"Error parsing line: {e}")
                    continue
    
    return total_api_calls, time_module_api_calls, matching_samples

def main():
    dataset_path = "/home/monoshi/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_new.jsonl"
    
    total_api, time_api, samples = count_time_module_api_calls(dataset_path)
    
    print(f"Total API call statements: {total_api}")
    print(f"API calls with time module functions: {time_api}")
    print(f"Percentage: {(time_api/total_api)*100:.2f}%" if total_api > 0 else "0%")
    
    print("\nSample statements with time module functions:")
    for i, sample in enumerate(samples[:10]):  # Show first 10 samples
        print(f"{i+1}. IDX: {sample['idx']}")
        print(f"   Statement: {sample['selected_statement']}")
        print(f"   Time functions: {sample['time_functions']}")
        print()

if __name__ == "__main__":
    main()

Total API call statements: 247
API calls with time module functions: 1
Percentage: 0.40%

Sample statements with time module functions:
1. IDX: 280
   Statement: _t = time.strptime(t, "%b %d %Y")
   Time functions: ['strptime']



In [8]:
import json

def print_dataset_fields(file_path):
    with open(file_path, 'r') as file:
        for line_num, line in enumerate(file, 1):
            try:
                data = json.loads(line.strip())
                
                print(f"=== idx ===")
                print(data.get('idx', 'N/A'))
                print("LOOP CODE:")
                print(data.get('loop_code', 'N/A'))
                print("\nQUESTION:")
                print(data.get('question', 'N/A'))
                print("\nANSWER:")
                print(data.get('answer', 'N/A'))
                print("\n" + "="*50 + "\n")
                
            except json.JSONDecodeError as e:
                print(f"Error parsing line {line_num}: {e}")
                continue

# Replace with your actual file path
file_path = "/home/monoshi/CodeSemantic/CodeSemantic/dataset/loop_iteration_dataset_python.jsonl"
print_dataset_fields(file_path)

=== idx ===
0
LOOP CODE:
1. def find(lst, key, value):
2.     for i, dic in enumerate(lst):
3.         if dic[key] == value:
4.             return i
5.     return None
6.
7. find([{'Variable': 'jenkins_admin_password', 'Type': 'password'}, {'Variable': 'ca_rootca_password', 'Type': 'password'}], 'Variable', 'something_not_there')

QUESTION:
How many times will the loop on line 2 execute when 'find([{'Variable': 'jenkins_admin_password', 'Type': 'password'}, {'Variable': 'ca_rootca_password', 'Type': 'password'}], 'Variable', 'something_not_there')' is called?

ANSWER:
2


=== idx ===
1
LOOP CODE:
1. def _global_import(name):
2.     p = __import__(name, globals(), locals(), level=1)
3.     lst = p.__all__ if '__all__' in dir(p) else dir(p)
4.     if lst:
5.         globals().pop(name, None)
6.         for k in lst:
7.             if not k.startswith('__'):
8.                 globals()[k] = p.__dict__[k]
9.                 __all__.append(k)
10.
11. _global_import('base')

QUESTION:
How m

In [7]:
import json
import pandas as pd

def analyze_statement_results_side_by_side(file_path):
    results = []
    
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line.strip())
            results.append(data)
    
    # Create a DataFrame for better visualization
    df_data = []
    for result in results:
        df_data.append({
            'Model': result['Model'],
            'Incontext': result['Incontext'],
            'Prompt': result['Prompt'],
            'CoT': result['CoT'],
            'shot': result['shot'],
            'quantization': result['quantization'],
            'accuracy': result['accuracy'],
            'accuracy_percent': f"{result['accuracy']*100:.2f}%"
        })
    
    df = pd.DataFrame(df_data)
    
    # Filter for shot 3, CoT: no, quantization: no
    filtered_df = df[
        (df['shot'] == 3) & 
        (df['CoT'] == 'no') & 
        (df['quantization'] == 'no')
    ]
    
    print("=== Model Comparison - Shot 3, CoT: no, Quantization: no ===\n")
    
    # Display results side by side
    same_results = filtered_df[filtered_df['Incontext'] == 'same']
    different_results = filtered_df[filtered_df['Incontext'] == 'different']
    
    print(f"{'Model':<35} {'Incontext: Same':<15} {'Incontext: Different':<20}")
    print("-" * 75)
    
    # Get all unique models
    all_models = set(same_results['Model']).union(set(different_results['Model']))
    
    for model in sorted(all_models):
        same_acc = same_results[same_results['Model'] == model]['accuracy_percent']
        diff_acc = different_results[different_results['Model'] == model]['accuracy_percent']
        
        same_display = same_acc.values[0] if not same_acc.empty else "N/A"
        diff_display = diff_acc.values[0] if not diff_acc.empty else "N/A"
        
        print(f"{model:<35} {same_display:<15} {diff_display:<20}")
    
    # Calculate and display differences
    print("\n" + "=" * 75)
    print("SUMMARY COMPARISON:\n")
    
    for model in sorted(all_models):
        same_row = same_results[same_results['Model'] == model]
        diff_row = different_results[different_results['Model'] == model]
        
        if not same_row.empty and not diff_row.empty:
            same_acc_val = same_row['accuracy'].values[0]
            diff_acc_val = diff_row['accuracy'].values[0]
            difference = same_acc_val - diff_acc_val
            better_worse = "SAME better" if difference > 0 else "DIFFERENT better"
            
            print(f"{model}:")
            print(f"  Same: {same_acc_val*100:.2f}% vs Different: {diff_acc_val*100:.2f}%")
            print(f"  Difference: {difference*100:+.2f} percentage points ({better_worse})")
            print()

# Run the analysis
analyze_statement_results_side_by_side('/home/monoshi/CodeSemantic/CodeSemantic/statement_Accuracy_Results/statement_results.jsonl')

=== Model Comparison - Shot 3, CoT: no, Quantization: no ===

Model                               Incontext: Same Incontext: Different
---------------------------------------------------------------------------
DeepSeek-Coder-V2-Lite-Instruct     51.93%          45.57%              
DeepSeek-R1-Distill-Llama-8B        41.10%          40.96%              
DeepSeek-R1-Distill-Qwen-14B        52.48%          51.29%              
DeepSeek-R1-Distill-Qwen-7B         43.49%          40.77%              
Llama-3.1-8B-Instruct               45.69%          41.88%              
Phi-3.5-mini-instruct               48.26%          33.39%              
Phi-4-mini-instruct                 46.79%          38.01%              
Qwen2.5-14B-Instruct-1M             56.70%          52.58%              
Qwen2.5-Coder-7B-Instruct           55.05%          46.31%              
anthropic.claude-3-5-sonnet-20241022-v2:0 53.06%          73.10%              
gemini-1.5-flash-002                71.50%          6

In [3]:
import json

def print_api_statements(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            # Parse JSON line
            data = json.loads(line.strip())
            
            # Check if it's an API statement type
            if data.get("Statement Type") == "API":
                print("=" * 80)
                print(f"Index: {data.get('idx', 'N/A')}")
                print("\n--- SOURCE CODE ---")
                print(data.get("Source Code", "N/A"))
                print("\n--- SELECTED STATEMENT ---")
                print(data.get("Selected Statement", "N/A"))
                print("\n--- FUNCTION INPUT ---")
                print(json.dumps(data.get("Function Input", {}), indent=2))
                print("\n--- VARIABLE VALUES BEFORE STATEMENT ---")
                print(json.dumps(data.get("Variable Values Before Statement", {}), indent=2))
                print("\n--- VALUE AFTER STATEMENT EXECUTION ---")
                print(data.get("Value After Statement Execution", "N/A"))
                print("=" * 80)
                print("\n")

if __name__ == "__main__":
    file_path = "/home/monoshi/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_new.jsonl"
    print_api_statements(file_path)

Index: 159

--- SOURCE CODE ---
def secrets_dir(env=os.getenv('D2_ENVIRONMENT', None),
                basedir=os.getenv('D2_SECRETS_BASEDIR', None)):
    if env is not None:
        env_str = str(env)
    else:
        cwd = os.getcwd()
        default_file = os.path.join(cwd, '.python_secrets_environment')
        if os.path.exists(default_file):
            with open(default_file, 'r') as f:
                env_str = f.read().strip()
        else:
            env_str = os.path.basename(cwd)
    if basedir is None:
        basedir = os.path.join(
                HOME,
                'secrets' if sys.platform.startswith('win') else '.secrets')
    return os.path.join(basedir, env_str)

secrets_dir(env=None, basedir=None)

--- SELECTED STATEMENT ---
default_file = os.path.join(cwd, '.python_secrets_environment')

--- FUNCTION INPUT ---
{
  "env": "None",
  "basedir": "None"
}

--- VARIABLE VALUES BEFORE STATEMENT ---
{
  "cwd": "'/local/rcs/jinjun/code/pytrace-collector/logs/pypibugs/